# 05 · Model Evaluation

In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from src.data_preprocessing import *
sns.set_theme(style="whitegrid")

In [ ]:
import joblib
from sklearn.metrics import classification_report
model = joblib.load('../models/churn_model.pkl')
X, y = prepare_xy(load_data())
_, X_test, _, y_test = split_data(X, y)
prob = model.predict_proba(X_test)[:, 1]
pred = (prob >= .5).astype(int)
print(classification_report(y_test, pred, target_names=['Retained','Churned']))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
ConfusionMatrixDisplay.from_predictions(y_test, pred);
RocCurveDisplay.from_predictions(y_test, prob);

In [ ]:
# Threshold trade-off
from sklearn.metrics import precision_score, recall_score
for t in [.3, .4, .5, .6]:
    p = (prob >= t).astype(int)
    print(f't={t}: precision={precision_score(y_test,p):.2f} recall={recall_score(y_test,p):.2f}')

In [ ]:
# Top drivers (logistic regression coefficients)
clf = model.named_steps['clf']
if hasattr(clf, 'coef_'):
    names = model.named_steps['prep'].get_feature_names_out()
    s = pd.Series(clf.coef_[0], index=names).sort_values()
    pd.concat([s.head(8), s.tail(8)]).plot.barh(figsize=(7,6));